# CVRIE
## Supervised training

**Summary :**
- Importing the dataset
- Cleaning the dataset
- ...

### Step 1: Importing the dataset from Kaggle.com

> Why did we choose this dataset ?

The [GlaS@MICCAI'2015: Gland Segmentation](https://www.kaggle.com/datasets/sani84/glasmiccai2015-gland-segmentation) dataset offered a lot of advantage, such as .... (labeled images, uniform data, clear and concise labels...)

In [149]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("sani84/glasmiccai2015-gland-segmentation")

print("Path to dataset files:", path)
os.rename(path, "/home/victor-ripert/delivery/barcelona/projects/AIA/CVRIE/dataset/gland-segmentation")
print("Now into the right folder")

100%|██████████| 172M/172M [00:14<00:00, 12.4MB/s] 

Extracting files...


Path to dataset files: /home/victor-ripert/.cache/kagglehub/datasets/sani84/glasmiccai2015-gland-segmentation/versions/1
Now into the right folder


### End of the Notebook
> Erasing the dataset

## Cleaning the Dataset

### Removing the unknown / unusable images

In this dataset, for each "normal" image, another dark image exist, and I don't really understand how can we used them. So to avoid unwanted interferences with the result of our model, I choose to removed them.

In [150]:
import os, glob

list_of_dataset_files = glob.glob('dataset/gland-segmentation/Warwick_QU_Dataset/*_anno.bmp', recursive=True)
for file in list_of_dataset_files :
    print("Removed file '{}' from dataset".format(file))
    try:
        os.remove(file)
    except OSError :
        print("Error while deleting file {}".format(file))

print("All unwanted file deleted")

Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/testA_27_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_63_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_74_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/testA_40_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_26_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_67_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_22_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/testA_55_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/train_46_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/testB_10_anno.bmp' from dataset
Removed file 'dataset/gland-segmentation/Warwick_QU_Dataset/testB_11_anno.bmp' f

### Resizing all the images at the same width and height.
Its important that all the images in the dataset have the same size, if not it could false the results.

We can observe just by looking at the dataset, that the images have more or less the sames size, so we're going to display the size of some images, to have an idea of the average size of the images in this dataset.

In [ ]:
import cv2
import os
from statistics import stdev

def get_image_size(filepath) :
    img = cv2.imread(filepath)
    a, b = img.shape[:2]
    return a,b

width_mean = 0
height_mean = 0

width_list = []
height_list = []

lowest_width = 10000
lowest_height = 10000
nb_normal_width = 0
nb_normal_heigth = 0

file_list = os.listdir('dataset/gland-segmentation/Warwick_QU_Dataset/')
nb_img = len(file_list) - 1 # Minus Grade.csv
for i in range(len(file_list)) :
    filepath = 'dataset/gland-segmentation/Warwick_QU_Dataset/{}'.format(file_list[i])
    if (os.path.exists(filepath) != True or file_list[i] == "Grade.csv") :
        continue
    width, height = get_image_size(filepath)
    if (width == 522) :
        nb_normal_width += 1
    if (height == 775) :
        nb_normal_heigth += 1
    width_mean += width
    height_mean += height
    width_list.append(width)
    height_list.append(height)
    if (width < lowest_width) :
        lowest_width = width
    if (height < lowest_height) :
        lowest_height = width

width_mean /= nb_img
height_mean /= nb_img
print("Mean of width and height :\t{},{}".format(width_mean, height_mean))
width_std_dev = stdev(width_list)
height_std_dev = stdev(height_list)
print("Standard deviation of width and height :\t{},{}".format(width_std_dev, height_std_dev))
print("Lowest width is {} and Lowest height is {}".format(lowest_width, lowest_height))
print("Amount of img having :\n\tWidth of 522: {}\n\tHeight of 775: {}".format(nb_normal_width, nb_normal_heigth))
print("Total amount of img : {}".format(nb_img))

Mean of width and height :	515.5090909090909,758.739393939394
Standard deviation of width and height :	21.550173498769123,53.606959068819236
Lowest width is 430 and Lowest height is 522
Amount of img having :
	Width of 522: 151
	Height of 775: 151
Total amount of img : 165


#### **Result :**
As we can see in our Dataset the average width of an image is `515.5` and the average heigth is `758.7`.
With a standard deviation of `21.5` for the width and `53.5` for the height.\
We can also observe that there is `151` images having a Width of `522` and also `151` having a Height of `775`. Over the `165` total images.

**So here we have two options :**
- **A** : Resize all the images to the lowest width and Height (which is `430` and `522`)
- **B** : Only resize the images that doesnt match a Width of `522` and a Height of `775`

I choose option **B**, because its more efficient to resize a lower amount of images (only `14`) than the other `151` ones, and because it don't interfere this much with the dataset.

In [ ]:
import sys

img_list = []
nb_resized_file = 0
for i in range(len(file_list)) :
    filepath = 'dataset/gland-segmentation/Warwick_QU_Dataset/{}'.format(file_list[i])
    if (os.path.exists(filepath) != True or file_list[i] == "Grade.csv") :
        continue
    img = cv2.imread(filepath)
    width, height = img.shape[:2]
    img_resized = cv2.resize(img, (430, 522), interpolation=cv2.INTER_LINEAR)
    img_list_tuple = (file_list[i], img_resized)
    img_list.append(img_list_tuple)
    print("Resized file {}".format(file_list[i]))
    nb_resized_file += 1

print("Total of resized files : {}".format(nb_resized_file))

Resized file train_17.bmp
Resized file train_26.bmp
Resized file train_2.bmp
Resized file testB_13.bmp
Resized file train_67.bmp
Resized file testA_11.bmp
Resized file train_22.bmp
Resized file train_57.bmp
Resized file train_65.bmp
Resized file testA_46.bmp
Resized file train_66.bmp
Resized file testA_23.bmp
Resized file train_79.bmp
Resized file testA_57.bmp
Resized file train_38.bmp
Resized file testB_18.bmp
Resized file train_63.bmp
Resized file testA_16.bmp
Resized file train_4.bmp
Resized file train_16.bmp
Resized file train_12.bmp
Resized file train_81.bmp
Resized file testB_17.bmp
Resized file testB_9.bmp
Resized file train_43.bmp
Resized file testA_36.bmp
Resized file train_82.bmp
Resized file train_50.bmp
Resized file testA_44.bmp
Resized file train_80.bmp
Resized file testA_52.bmp
Resized file testA_4.bmp
Resized file train_10.bmp
Resized file testA_38.bmp
Resized file testA_20.bmp
Resized file testA_42.bmp
Resized file testA_55.bmp
Resized file train_30.bmp
Resized file tes

##  Building the dataframe

Now that our dataset is cleaned and all the images resized, we cant to load out data from `Grade.csv` and link each rows with is corresponding image.

For this we will first load the dataframe from the csv, then we will select the column we keep of throw. Then we have to transform each loaded image into a numpy array. And finally linked all the rows together.

In [153]:
import pandas as pd

dataframe = pd.read_csv('dataset/gland-segmentation/Warwick_QU_Dataset/Grade.csv')
display(dataframe)

,name,patient ID,grade (GlaS),grade (Sirinukunwattana et al. 2015)
0,testA_1,4,benign,adenomatous
1,testA_10,10,benign,healthy
2,testA_11,9,benign,healthy
3,testA_12,11,malignant,poorly differentiated
4,testA_13,7,malignant,moderately differentiated
...,...,...,...,...
160,train_82,2,malignant,moderately-to-poorly differentated
161,train_83,11,malignant,poorly differentiated
162,train_84,15,benign,healthy
163,train_85,10,benign,healthy


### Select the column we keep

For this model I choose to use the GlaS grade which have only two label (`malignant` or `benign`)

Also we have to "encode" the label" for the model to use it. `malignant` will become 1 and `benign` will become 0.

In [154]:
dataframe.drop('patient ID', axis=1, inplace=True)
dataframe.drop(dataframe.columns[2], axis=1, inplace=True)
dataframe.columns = ['name', 'status']
display(dataframe)
df = pd.get_dummies(dataframe, columns=['status'], dtype='int', drop_first=True)
display(df)

,name,status
0,testA_1,benign
1,testA_10,benign
2,testA_11,benign
3,testA_12,malignant
4,testA_13,malignant
...,...,...
160,train_82,malignant
161,train_83,malignant
162,train_84,benign
163,train_85,benign


,name,status_ malignant
0,testA_1,0
1,testA_10,0
2,testA_11,0
3,testA_12,1
4,testA_13,1
...,...,...
160,train_82,1
161,train_83,1
162,train_84,0
163,train_85,0


### Link each row to the corresponding image from the dataset

We're going to use the `name` column of the dataframe.

In [ ]:
cols = list(df.columns)

display(list(df.columns))

matched_image = {}

for index, row in df.iterrows() :
    row_name = row['name']
    for img_name, img in img_list :
        tmp_name = "{}.bmp".format(row_name)
        if tmp_name == img_name :
            print("Match")
            tmp_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            matched_image[row_name] = tmp_image.flatten()

df['image'] = df['name'].map(matched_image)
display(df)

['name', 'status_ malignant']

Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match
Match


,name,status_ malignant,image
0,testA_1,0,"[252, 238, 240, 246, 226, 242, 234, 207, 237, ..."
1,testA_10,0,"[222, 151, 218, 211, 133, 209, 211, 124, 211, ..."
2,testA_11,0,"[218, 87, 189, 214, 80, 195, 230, 95, 193, 234..."
3,testA_12,1,"[130, 46, 154, 136, 50, 157, 148, 61, 162, 149..."
4,testA_13,1,"[148, 37, 135, 148, 40, 142, 150, 39, 144, 147..."
...,...,...,...
160,train_82,1,"[158, 69, 157, 170, 76, 162, 163, 72, 155, 130..."
161,train_83,1,"[240, 183, 239, 240, 182, 236, 239, 175, 223, ..."
162,train_84,0,"[221, 130, 219, 222, 117, 218, 253, 162, 251, ..."
163,train_85,0,"[203, 79, 162, 212, 78, 175, 209, 84, 167, 200..."


### Separation of the dataset

Now that each row of the dataframe is linked to is corresponding image (in form of a numpy.nparray)
we can separate the dataframe between the training part and the testin part.

For this we need first to know the shape of our dataframe and separate the input X (the image) from the outupn y (the label)

In [156]:
print(df.shape)

(165, 3)


In [163]:
import numpy as np

print("rows:", len(df))
print("missing image:", df["image"].isna().sum())

is_array = df["image"].apply(lambda x: isinstance(x, np.ndarray))
print("non-ndarray:", (~is_array).sum())

tmp = df.loc[is_array, "image"].apply(lambda x: x.shape)
print("unique shapes:", tmp.value_counts().head(10))

tmp_len = df.loc[is_array, "image"].apply(lambda x: x.size)
print("unique lengths:", tmp_len.value_counts().head(10))

rows: 165
missing image: 0
non-ndarray: 0
unique shapes: image
(1213650,)    151
(673380,)      14
Name: count, dtype: int64
unique lengths: image
1213650    151
673380      14
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# keep only valid ndarray images
df = df[df["image"].notna()].copy()
df = df[df["image"].apply(lambda x: isinstance(x, np.ndarray))].copy()

# enforce same feature length
lengths = df["image"].apply(lambda x: x.size)
target_len = lengths.mode().iloc[0]   # most common length
df = df[lengths == target_len].copy()

X = np.stack(df["image"].to_numpy()).astype(np.float32) / 255.0
y = df[df.columns[1]].to_numpy().astype(np.int32).ravel()

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=104, test_size=0.40, shuffle=True)

X shape: (151, 1213650)
y shape: (151,)


In [158]:
print('X_train : ')
print(X_train.head())
print(X_train.shape)

print('')
print('X_test : ')
print(X_test.head())
print(X_test.shape)

print('')
print('y_train : ')
print(y_train.head())
print(y_train.shape)

print('')
print('y_test : ')
print(y_test.head())
print(y_test.shape)

X_train : 
134    [221, 103, 189, 229, 111, 196, 229, 109, 188, ...
102    [89, 22, 125, 89, 21, 121, 95, 28, 124, 97, 33...
106    [108, 23, 98, 115, 32, 100, 114, 28, 109, 114,...
65     [252, 246, 249, 253, 247, 251, 252, 245, 248, ...
55     [242, 145, 210, 198, 116, 189, 176, 75, 178, 2...
Name: image, dtype: object
(99,)

X_test : 
152    [248, 242, 242, 253, 247, 247, 254, 248, 248, ...
46     [235, 138, 205, 246, 149, 210, 234, 145, 217, ...
70     [181, 90, 216, 173, 101, 209, 159, 93, 197, 13...
20     [198, 133, 191, 197, 137, 201, 210, 154, 216, ...
34     [218, 137, 216, 227, 146, 216, 233, 149, 216, ...
Name: image, dtype: object
(66,)

y_train : 
134    0
102    1
106    0
65     1
55     0
Name: status_ malignant, dtype: int64
(99,)

y_test : 
152    1
46     1
70     1
20     0
34     0
Name: status_ malignant, dtype: int64
(66,)


### Training !!

Now that our dataframe is separated between input and output for both Train and Test. We can create our model ans start to train it.

I choose to use a model of type SVM.

SVM stand for **Suppot Vector Machine**. It is a supervised learning algorithm often used for classification....

In [ ]:
from sklearn import svm
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

model = svm.SVC(kernel='linear', C=1.0)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_pred, y_test)
print(f"The model is {accuracy*100}% accurate")
print("Predictions of X_test :")
display(y_pred)
print("Excepted (from y_test) :")
display(y_test)

The model is 60.65573770491803% accurate
Predictions of X_test :


array([1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1], dtype=int32)

Excepted (from y_test) :


array([1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1], dtype=int32)

In [167]:
import shutil
shutil.rmtree("/home/victor-ripert/delivery/barcelona/projects/AIA/CVRIE/dataset/gland-segmentation")
print("Dataset Removed !")

Dataset Removed !
